### Some AI-experimentation to replace dataprep (COMPLETELY OPTIONAL)

Feel free to modify, experiment, study etc.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# open the data as a pandas DataFrame
df = pd.read_csv("../../houses.csv")

def dataset_insights(df):
    insights = []

    for col in df.columns:
        s = df[col]

        # Missing
        missing = s.isna().mean()
        if missing > 0.01:
            insights.append(
                f"{col}: {missing:.1%} missing values"
            )

        # Numeric diagnostics
        if pd.api.types.is_numeric_dtype(s):
            x = s.dropna()

            if len(x) > 7:
                skew = stats.skew(x)

                if abs(skew) > 1:
                    direction = "right" if skew > 0 else "left"
                    insights.append(
                        f"{col}: strongly {direction}-skewed "
                        f"(skewness={skew:.2f})"
                    )

                normal = stats.normaltest(x)

                if normal.pvalue < 0.001:
                    insights.append(
                        f"{col}: strongly deviates from normality"
                    )

            zero_pct = (x == 0).mean()
            if zero_pct > 0.05:
                insights.append(
                    f"{col}: {zero_pct:.1%} zeros"
                )

            negative_pct = (x < 0).mean()
            if negative_pct > 0.01:
                insights.append(
                    f"{col}: {negative_pct:.1%} negative values"
                )

            # IQR outliers
            q1, q3 = x.quantile([.25, .75])
            iqr = q3 - q1
            outliers = ((x < q1 - 1.5 * iqr) |
                        (x > q3 + 1.5 * iqr)).sum()

            if outliers:
                insights.append(
                    f"{col}: {outliers:,} IQR outliers"
                )

        # Categorical diagnostics
        else:
            nunique = s.nunique(dropna=True)

            if nunique == len(s.dropna()):
                insights.append(
                    f"{col}: all non-null values are unique"
                )

            if nunique > 50:
                insights.append(
                    f"{col}: high cardinality ({nunique:,} values)"
                )

    return insights

In [8]:
dataset_insights(df)

['id: strongly deviates from normality',
 'date: high cardinality (372 values)',
 'price: strongly right-skewed (skewness=4.02)',
 'price: strongly deviates from normality',
 'price: 1,146 IQR outliers',
 'bedrooms: strongly right-skewed (skewness=1.97)',
 'bedrooms: strongly deviates from normality',
 'bedrooms: 546 IQR outliers',
 'bathrooms: strongly deviates from normality',
 'bathrooms: 571 IQR outliers',
 'sqft_living: strongly right-skewed (skewness=1.47)',
 'sqft_living: strongly deviates from normality',
 'sqft_living: 572 IQR outliers',
 'sqft_lot: strongly right-skewed (skewness=13.06)',
 'sqft_lot: strongly deviates from normality',
 'sqft_lot: 2,425 IQR outliers',
 'floors: strongly deviates from normality',
 'waterfront: strongly right-skewed (skewness=11.38)',
 'waterfront: strongly deviates from normality',
 'waterfront: 99.2% zeros',
 'waterfront: 163 IQR outliers',
 'view: strongly right-skewed (skewness=3.40)',
 'view: strongly deviates from normality',
 'view: 90.2%

In [9]:
import numpy as np
import pandas as pd

from itertools import combinations
from scipy.stats import ks_2samp, wasserstein_distance


def similar_distributions(
    df,
    columns=None,
    threshold=0.2,
    min_samples=20,
    numeric_only=True,
):
    """
    Find pairs of variables with similar empirical distributions.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.

    columns : list[str], optional
        Columns to compare. If None, all suitable columns are used.

    threshold : float, default=0.2
        Maximum combined similarity score to report.
        Lower = more similar.

    min_samples : int, default=20
        Minimum number of non-null observations required per variable.

    numeric_only : bool, default=True
        If True, only numeric columns are considered.

    Returns
    -------
    pd.DataFrame
        Ranked pairs of variables with similarity metrics.
    """

    if columns is None:
        if numeric_only:
            columns = df.select_dtypes(include="number").columns.tolist()
        else:
            columns = df.columns.tolist()

    # Remove unsuitable columns
    data = {}

    for col in columns:
        x = pd.to_numeric(df[col], errors="coerce").dropna()

        if len(x) >= min_samples and x.nunique() > 1:
            data[col] = x.to_numpy()

    results = []

    for col1, col2 in combinations(data, 2):
        x = data[col1]
        y = data[col2]

        # KS test
        ks_stat, ks_p = ks_2samp(x, y)

        # Wasserstein distance
        wd = wasserstein_distance(x, y)

        # Normalize Wasserstein by pooled standard deviation.
        # This makes the value somewhat comparable across variables
        # with different scales.
        pooled_std = np.std(np.concatenate([x, y]))

        if pooled_std > 0:
            wd_normalized = wd / pooled_std
        else:
            wd_normalized = 0.0

        # Combined score.
        #
        # KS is already in [0, 1].
        # Normalize Wasserstein using 1 - exp(-distance), which
        # keeps it in [0, 1] without imposing a hard cutoff.
        wd_score = 1 - np.exp(-wd_normalized)

        score = 0.5 * ks_stat + 0.5 * wd_score

        if score <= threshold:
            results.append({
                "variable_1": col1,
                "variable_2": col2,
                "similarity_score": score,
                "ks_statistic": ks_stat,
                "ks_pvalue": ks_p,
                "wasserstein": wd,
                "wasserstein_normalized": wd_normalized,
            })

    if not results:
        return pd.DataFrame(
            columns=[
                "variable_1",
                "variable_2",
                "similarity_score",
                "ks_statistic",
                "ks_pvalue",
                "wasserstein",
                "wasserstein_normalized",
            ]
        )

    return (
        pd.DataFrame(results)
        .sort_values("similarity_score")
        .reset_index(drop=True)
    )

similar_distributions(df)

,variable_1,variable_2,similarity_score,ks_statistic,ks_pvalue,wasserstein,wasserstein_normalized
0,sqft_lot,sqft_lot15,0.050995,0.034886,7.324228e-12,2437.990006,0.069462
1,sqft_living,sqft_living15,0.130534,0.069079,2.789804e-45,173.030861,0.213181
2,waterfront,yr_renovated,0.148474,0.042289,3.138095e-17,84.394716,0.293914
3,view,yr_renovated,0.155285,0.055985,7.021471e-30,84.361172,0.293813
4,bedrooms,condition,0.191324,0.128164,3.357728e-155,0.235784,0.293680
